# Module 3.3 — Mô hình Random Forest (Cửa hàng × Nhóm hàng × Ngày)

Dự báo `so_luong` mỗi ngày cho từng cặp cửa hàng × nhóm hàng.

- **Đầu vào:** `outputs/daily_store_category.csv` + đặc trưng từ `tao_dac_trung` (lag, rolling dời 1 ngày + lịch + mã cửa hàng/nhóm hàng/vùng; hàm tự kiểm tra rò rỉ).
- Mốc chia, chỉ số đánh giá và baseline lấy từ module dùng chung (`chia_tap`, `danh_gia`, `bang_baseline`) để so sánh công bằng với các mô hình khác.
- Tối ưu trên Validation, train lại trên toàn bộ Train, chấm trên Test (tháng 3) cùng các baseline Naive, Seasonal Naive, MA7.
- Mục 5 — kết quả và thảo luận, cùng khung với các mô hình khác (xem *Chuẩn đầu ra mỗi mô hình* trong `README.md`).
- **Đầu ra** `outputs/rf/`: `rf_tuning.csv`, `rf_best_params.json`, `rf_metrics.csv`, `rf_model.joblib`, `rf_predictions.csv`, `rf_feature_importance.*`, `rf_error_by_category.csv`, `rf_error_by_store.csv`, `rf_thuc_te_vs_du_bao.png`.

**Thứ tự chạy:** `Module3_1_tien_xu_ly` → `Module3_2_eda` → `Module3_3_random_forest`. Notebook 2 và 3 đọc `outputs/daily_store_category.csv`; nếu chưa có thì tự dựng lại từ file gốc.

**Cấu hình** (`.env`, mẫu ở `.env.example`): `DATA_FILE`, `DRIVE_ID`, `OUT_DIR`, `KHU_VUC`. Nếu chưa có file dữ liệu, module tự tải từ Google Drive theo `DRIVE_ID`.

**Chạy trên Google Colab:** upload `module3_preprocess.py` và `.env` lên cùng thư mục với notebook, rồi Runtime → Run all.

In [ ]:
import os, json, time, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from module3_preprocess import (OUT_DIR as OUT, SEED, doc_bang_ngay, tao_dac_trung, chia_tap, danh_gia,
                                bang_baseline, loi_theo_nhom, nop_ket_qua, tai_ve)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
daily = doc_bang_ngay()
print(f"Bảng mô hình: {len(daily):,} dòng")

## 1. Đặc trưng và chia tập

Dùng nguyên `FEATS` của module chung (lag, rolling dời 1 ngày, lịch, mã cửa hàng/nhóm hàng/vùng dạng số). Baseline được chấm trên cả Validation (để so khi tối ưu) và Test.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterSampler

N_ITER = 12                              # số bộ tham số thử khi tối ưu
OUT_RF = f"{OUT}/rf"; os.makedirs(OUT_RF, exist_ok=True)
d, FEATS = tao_dac_trung(daily)
fit, val, tr, te = chia_tap(d)

In [ ]:
base_val = pd.DataFrame(bang_baseline(val)).T
rows = bang_baseline(te)
print("Baseline trên Validation:"); display(base_val)
print("Baseline trên Test:"); display(pd.DataFrame(rows).T)

## 2. Tối ưu tham số trên Validation

- 12 cấu hình ngẫu nhiên (`ParameterSampler`, `random_state=42`) trên `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`, `criterion` (`squared_error` / `poisson`).
- Train trên 15/01–14/02, chấm trên Validation 15–28/02; **chọn cấu hình theo MAE trên Validation** — cùng tiêu chí với các mô hình khác.

In [ ]:
space = {"n_estimators": [150, 300], "max_depth": [8, 12, 16, 24, None], "min_samples_leaf": [1, 3, 5, 10, 20],
         "max_features": ["sqrt", 0.5, 0.8, 1.0], "criterion": ["squared_error", "poisson"]}
params, log = list(ParameterSampler(space, n_iter=N_ITER, random_state=SEED)), []
for i, p in enumerate(params):
    t = time.time()
    m = RandomForestRegressor(random_state=SEED, n_jobs=-1, **p).fit(fit[FEATS], fit["so_luong"])
    s = danh_gia(val["so_luong"], m.predict(val[FEATS]))
    log.append({"id": i, **p, **s, "giây": round(time.time() - t, 1)})
    print(f"{i + 1:2}/{N_ITER} MAE={s['MAE']:.4f} WAPE={s['WAPE']:.3f} {p}")
tuning = pd.DataFrame(log).sort_values("MAE").reset_index(drop=True)
tuning.to_csv(f"{OUT_RF}/rf_tuning.csv", index=False, encoding="utf-8-sig")
best = params[tuning.loc[0, "id"]]
with open(f"{OUT_RF}/rf_best_params.json", "w") as f:
    json.dump(best, f, indent=2)
print("Tốt nhất:", best)

## 3. Train lại trên toàn bộ Train và chấm Test

Train lại cấu hình tốt nhất trên 15/01–28/02 rồi chấm trên tháng 3. Dự báo âm được cắt về 0.

In [ ]:
rf = RandomForestRegressor(random_state=SEED, n_jobs=-1, **best).fit(tr[FEATS], tr["so_luong"])
te["du_bao_rf"] = np.clip(rf.predict(te[FEATS]), 0, None)
rows["Random Forest"] = danh_gia(te["so_luong"], te["du_bao_rf"])
result = pd.DataFrame(rows).T
result["WAPE so với MA7"] = result["WAPE"] / result.loc["MA7", "WAPE"] - 1
result.to_csv(f"{OUT_RF}/rf_metrics.csv", encoding="utf-8-sig")
joblib.dump(rf, f"{OUT_RF}/rf_model.joblib", compress=3)
display(result)

## 4. Đặc trưng quan trọng và lỗi theo nhóm hàng / cửa hàng / ngày

In [ ]:
imp = pd.Series(rf.feature_importances_, index=FEATS).sort_values()
imp.to_csv(f"{OUT_RF}/rf_feature_importance.csv", header=["importance"], encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(7, 5)); ax.barh(imp.index, imp.values, color="#0d9488")
ax.set_title("Random Forest — mức độ quan trọng của đặc trưng"); fig.tight_layout()
fig.savefig(f"{OUT_RF}/rf_feature_importance.png", dpi=150); plt.show()

err_cat = loi_theo_nhom(te, "du_bao_rf", "Nhóm hàng")
err_store = loi_theo_nhom(te, "du_bao_rf", ["Mã cửa hàng", "Khu vực"])
err_cat.to_csv(f"{OUT_RF}/rf_error_by_category.csv", encoding="utf-8-sig")
err_store.to_csv(f"{OUT_RF}/rf_error_by_store.csv", encoding="utf-8-sig")
display(err_cat)
print(f"Random Forest có WAPE thấp hơn MA7 ở {(err_cat.WAPE < err_cat.WAPE_MA7).sum()}/{len(err_cat)} nhóm hàng "
      f"và {(err_store.WAPE < err_store.WAPE_MA7).sum()}/{len(err_store)} cửa hàng")

day = te.groupby("Ngày")[["so_luong", "du_bao_rf", "tb_7"]].sum()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(day.index, day["so_luong"], color="#1f2a2a", lw=2, label="Thực tế")
ax.plot(day.index, day["du_bao_rf"], color="#0d9488", lw=2, label="Random Forest")
ax.plot(day.index, day["tb_7"], color="#eb6834", lw=1.5, ls="--", label="MA7")
ax.set_title("Tổng số lượng tháng 3 — thực tế và dự báo"); ax.legend(); ax.grid(axis="y", alpha=.3)
fig.tight_layout(); fig.savefig(f"{OUT_RF}/rf_thuc_te_vs_du_bao.png", dpi=150); plt.show()
te[["Mã cửa hàng", "Nhóm hàng", "Khu vực", "Vùng", "Ngày", "so_luong", "du_bao_rf", "lag_1", "lag_7", "tb_7"]] \
    .to_csv(f"{OUT_RF}/rf_predictions.csv", index=False, encoding="utf-8-sig")

## 5. Kết quả và thảo luận

*Số liệu từ lần chạy ngày 09/10/2026 trên bảng mô hình chưa lọc (giữ bán buôn, online, trả lại) — cùng bảng với CatBoost. Chạy lại sau khi nhóm chốt bộ lọc thì cập nhật lại mục này.*

**Tối ưu tham số.** Cấu hình tốt nhất trên Validation: `n_estimators=300`, `max_depth=8`, `min_samples_leaf=20`, `max_features="sqrt"`, `criterion="squared_error"` — MAE Validation 0,868, **vẫn kém MA7 (0,847)**; không cấu hình nào trong 12 cấu hình thắng MA7 trên Validation. Cây càng bị giới hạn càng tốt: 3 cấu hình đầu đều có `min_samples_leaf=20` và `max_features="sqrt"`, các cấu hình `min_samples_leaf=1` cho MAE ≈ 1,00. `criterion` ít ảnh hưởng (`poisson` 0,872 so với `squared_error` 0,868).

| Cấu hình (3 tốt nhất theo MAE Val) | MAE Val | RMSE Val | WAPE Val | Bias Val |
|---|---:|---:|---:|---:|
| leaf 20 · sqrt · depth 8 · squared_error | **0,868** | 7,807 | **0,767** | +18,2% |
| leaf 20 · sqrt · depth 8 · poisson | 0,872 | **7,753** | 0,771 | +20,3% |
| leaf 20 · sqrt · depth 24 · poisson | 0,879 | 7,757 | 0,778 | +22,4% |
| *MA7 (tham chiếu)* | *0,847* | *8,474* | *0,749* | *+11,2%* |

**Kết quả trên Test (tháng 3, 67.146 ô):**

| Mô hình | MAE | RMSE | WAPE | R² | Bias |
|---|---:|---:|---:|---:|---:|
| Naive (lag 1) | 0,887 | 6,675 | 0,802 | −0,142 | −0,9% |
| Seasonal Naive (lag 7) | 0,908 | 6,728 | 0,821 | −0,160 | +1,6% |
| MA7 | **0,744** | 4,975 | **0,674** | 0,365 | **−0,2%** |
| **Random Forest** | 0,757 | **4,743** | 0,686 | **0,423** | +0,8% |

**Thảo luận.**

1. **Random Forest chưa thắng MA7 trên tiêu chí chọn mô hình** — MAE 0,757 so với 0,744, WAPE kém 1,8% (0,686 so với 0,674). Ngược lại RF tốt nhất về RMSE (4,743) và R² (0,423 so với 0,365 của MA7): sai số ở các ô bán lớn nhỏ hơn.
2. **Đúng tổng lượng.** Bias +0,8%: trung bình 2.413 sản phẩm/ngày toàn hệ thống so với thực tế 2.393 (MA7 2.388). Ở 278 ô bán ≥ 20 sản phẩm/ngày (thực tế 13.570), RF dự báo 10.622 — gần MA7 (10.904), khác hẳn CatBoost (3.022).
3. **Thắng MA7 ở 4/23 nhóm hàng, đúng là 4 nhóm lớn nhất** — Điện thoại (WAPE 0,533 so với 0,559), Phụ kiện (0,498 so với 0,507), Hcare (0,547 so với 0,552), Tai nghe (0,841 so với 0,845), chiếm 81% lượng bán tháng 3 — nhưng chỉ 22/130 cửa hàng. Ở 19 nhóm còn lại RF dự báo dư 15% (16.129 so với thực tế 14.058; MA7 14.073), làm tổng sai số tuyệt đối lớn hơn MA7 1.774 sản phẩm, nhiều hơn phần thắng 891 ở 4 nhóm lớn → WAPE tổng kém MA7.
4. **Đặc trưng quan trọng:** trung bình 14 ngày (21,3%), trung bình 7 ngày (20,1%), độ lệch chuẩn 14 ngày (16,1%), lag 1 (14,5%), độ lệch chuẩn 7 ngày (11,7%) — mô hình gần như chỉ dựa vào mức và độ biến động bán gần đây. Lịch và mã cửa hàng/nhóm hàng/vùng cộng lại chưa tới 4% (`ngay_trong_thang` 1,6%), khác CatBoost (`ngay_trong_thang` 12,5%).
5. **So với CatBoost** (cùng bảng, đặc trưng, mốc chia): CatBoost thắng MA7 về MAE/WAPE nhưng dự báo thiếu 42% tổng lượng và thua ở Điện thoại, Phụ kiện; RF giữ đúng tổng lượng và thắng ở chính hai nhóm đó nhưng thua ở nhóm thưa. Nếu nhóm chốt điều kiện |Bias| ≤ 10%, RF là mô hình học máy duy nhất hiện đạt điều kiện, nhưng vẫn chưa thắng MA7 về MAE/WAPE.
6. **R² âm của Naive / Seasonal Naive không phải lỗi tính.** 10 ô chiếm 79% tổng bình phương sai số của Naive; riêng HN007 (online, đơn bán buôn lớn) chiếm 76%. Lặp lại một ô bán buôn đột biến của hôm trước tạo hai sai số lớn, nên tệ hơn dự báo bằng trung bình. Nếu bỏ bán buôn (`tao_bang_ngay(..., bo_ban_buon=True)`), R² của cả 3 baseline trên Test đều dương (Naive 0,496 · Seasonal Naive 0,529 · MA7 0,700) — chờ nhóm chốt bộ lọc.

**Hướng cải thiện:** cấu hình tốt nhất nằm ở biên không gian tìm kiếm (`min_samples_leaf` lớn nhất, `max_depth` nhỏ nhất) → thử cây đơn giản hơn (`min_samples_leaf` 20–100, `max_depth` 4–8).

**Giới hạn:** bảng mô hình chưa lọc bán buôn/online, chỉ 12 cấu hình, một mốc chia; Validation 15–28/02 không đại diện cho Test (R² của MA7: 0,124 trên Validation, 0,365 trên Test).

## Nộp kết quả vào báo cáo và tải về (Colab)

`nop_ket_qua` lọc các kết quả đưa vào báo cáo sang `bao_cao/du_lieu/rf/`. Sau đó chạy `python module5/tao_bao_cao.py` để cập nhật `bao_cao/BAO_CAO.md` rồi commit `bao_cao/` (trên Colab: giải nén `bao_cao_rf.zip` vào `bao_cao/du_lieu/rf/` trong repo trước).

In [ ]:
nop_ket_qua("rf", val)
tai_ve(OUT_RF, "module3_rf_outputs")